In [1]:
# ── Cell 1 — imports, path & reload ───────────────────────────────────────────
import sys, os, importlib, warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
sys.path.insert(0, os.path.abspath('../../model'))
sys.path.insert(0, os.path.abspath('.'))
sys.path.insert(0, os.path.abspath('../.'))

import numpy as np
import matplotlib.pyplot as plt
import xso
from xso.parscans import run_xso_parscan, run_xso_stabilityscan, avg_tail

import cariaco_obs as co
import cariaco_baseline_setups as cbs
import parscan_utils as pu
import parscan_plots as pp
for m in (co, cbs, pu, pp):            # pick up edits without a kernel restart
    importlib.reload(m)

phyto_esd, zoo_esd = cbs.phyto_esd, cbs.zoo_esd
print(f'grid: {len(phyto_esd)} phyto ({phyto_esd[0]:.2g}–{phyto_esd[-1]:.0f} µm), '
      f'{len(zoo_esd)} zoo ({zoo_esd[0]:.2g}–{zoo_esd[-1]:.0f} µm)')

grid: 40 phyto (0.2–200 µm), 40 zoo (2–2000 µm)


In [2]:
# ── RUNG 1: recycle_fraction sweep × F_N — does export kill the instability? ──
import importlib, numpy as np, xso
import baseline_r0_testing_comps as r0c, baseline_r0_testing_setups as r0s
import cariaco_obs as co, parscan_utils as pu
for m in (r0c, r0s): importlib.reload(m)
N = r0s.N_CLASSES
bin_defs = co.TARGET_BIN_DEFINITIONS
W = pu.phyto_bin_matrix(r0s.phyto_esd, bin_defs); geomeans = pu.CARIACO_PHYTO_BIN_GEOMEANS
TAIL = 1000
*_, mdf, fc_up = co.load_cariaco_targets('upwelling', months='hplc', agg='median', include_temperature=True)
pcols = [b['column'] for b in bin_defs if b['type']=='phyto']
obs_f = np.nanmedian(mdf[pcols].dropna().values / mdf[pcols].dropna().values.sum(1,keepdims=True), 0)

# RK45 + looser instability floor, passed at CALL time (run-specific setup; not baked)
test_setup = xso.setup(solver='solve_ivp', model=r0s.model_r0, time=np.arange(0, 8000, 1.0),
                       input_vars=r0s.make_r0_input_vars(), output_vars=r0s.SLIM_OUTPUT_VARS,
                       solver_kwargs={**r0s.IVP_SOLVER_KWARGS, 'instability_neg_threshold': -1e-3})
rf_ov = lambda rf: {'GrazingRouter__recycle_fraction': rf,
                    'PhytoLoss__recycle_fraction': rf, 'ZooLoss__recycle_fraction': rf}

print(f"obs upwelling Pico/Nano/Micro = {np.round(obs_f,3)}")
print(f"{'F_N':>5}{'recyc':>7}{'Pico':>7}{'Nano':>7}{'Micro':>7}{'mcs':>7}{'CV':>9}{'status':>8}")
for fn in [2.58, 6.0]:
    for rf in [0.0, 0.25, 0.5, 0.75, 1.0]:
        ov = {'Inflow__de': fc_up['Inflow__de'], 'Temperature__value': fc_up['Temperature__value'],
              'Inflow__FN': fn, **rf_ov(rf)}
        out = pu.run_single_point(r0s.model_r0, test_setup, scan_params={}, fixed_overrides=ov)
        P = out['Phytoplankton__biomass'].values
        if np.isnan(P).any():
            print(f"{fn:5.2f}{rf:7.2f}{'':>28}     NaN"); continue
        Pt = P[:, -TAIL:]; f=(W@Pt.mean(1)); f=f/f.sum(); cv=Pt.sum(0).std()/abs(Pt.sum(0).mean())
        print(f"{fn:5.2f}{rf:7.2f}{f[0]:7.3f}{f[1]:7.3f}{f[2]:7.3f}"
              f"{10**(np.log10(geomeans)@f):7.2f}{cv:9.4f}{'ok':>8}")
    print('-'*51)

obs upwelling Pico/Nano/Micro = [0.448 0.224 0.313]
  F_N  recyc   Pico   Nano  Micro    mcs       CV  status
 2.58   0.00  1.000  0.000  0.000   0.63   0.0005      ok
 2.58   0.25  1.000  0.000  0.000   0.63   0.0007      ok
 2.58   0.50  1.000  0.000  0.000   0.63   0.0006      ok
 2.58   0.75  1.000  0.000  0.000   0.63   0.0006      ok
 2.58   1.00  0.716  0.085  0.199   1.91   0.0279      ok
---------------------------------------------------
 6.00   0.00  1.000  0.000  0.000   0.63   0.0007      ok
 6.00   0.25  1.000  0.000  0.000   0.63   0.0007      ok


KeyboardInterrupt: 

In [ ]:
# ── sinking magnitude × recycle at regime F_N — does sinking set the knife-edge? ──
WSINKS = [0.0, 1.0, 2.5, 5.0, 10.0]   # m/d  (5.0 = current)
RFS    = [0.0, 0.5, 1.0]
fnreg  = 2.58
print(f"{'w_sink':>7}{'recyc':>7}{'Pico':>7}{'Nano':>7}{'Micro':>7}{'mcs':>7}{'sinkP/P':>9}{'CV':>9}{'stat':>6}")
for ws in WSINKS:
    for rf in RFS:
        ov = {'Inflow__de': fc_up['Inflow__de'], 'Temperature__value': fc_up['Temperature__value'],
              'Inflow__FN': fnreg, 'PhytoSinking__w_sink': ws, **rf_ov(rf)}
        out = pu.run_single_point(r0s.model_r0, test_setup, scan_params={}, fixed_overrides=ov)
        P = out['Phytoplankton__biomass'].values
        if np.isnan(P).any():
            print(f"{ws:7.1f}{rf:7.2f}{'  NaN':>50}"); continue
        Pt = P[:, -TAIL:]; f=(W@Pt.mean(1)); f=f/f.sum(); cv=Pt.sum(0).std()/abs(Pt.sum(0).mean())
        print(f"{ws:7.1f}{rf:7.2f}{f[0]:7.3f}{f[1]:7.3f}{f[2]:7.3f}"
              f"{10**(np.log10(geomeans)@f):7.2f}{ws/fc_up['Inflow__de']:9.3f}{cv:9.4f}{'ok':>6}")
print('-'*64)